In [23]:
!pip install -q groq requests beautifulsoup4

In [24]:
from groq import Groq
from getpass import getpass

# Enter your Groq API key
GROQ_API_KEY = getpass("Enter your Groq API Key: ")

client = Groq(api_key=GROQ_API_KEY)

# Check which models are available to your account
available_models = {
    model.id for model in client.models.list().data
}

preferred_models = [
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant"
]

MODEL_NAME = next(
    (model for model in preferred_models if model in available_models),
    None
)

if MODEL_NAME is None:
    raise RuntimeError(
        "No preferred chat model found. Check your Groq model catalogue."
    )

print("Groq connected successfully!")
print("Selected model:", MODEL_NAME)

Enter your Groq API Key: ··········
Groq connected successfully!
Selected model: openai/gpt-oss-120b


In [25]:
import sqlite3

conn = sqlite3.connect("business.db")
conn.execute("PRAGMA foreign_keys = ON")

cursor = conn.cursor()

cursor.executescript("""
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS employees;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS departments;

CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL
);

CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    employee_name TEXT NOT NULL,
    salary REAL,
    department_id INTEGER,
    FOREIGN KEY (department_id)
        REFERENCES departments(department_id)
);

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    city TEXT
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    amount REAL,
    order_date TEXT,
    FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id)
);

INSERT INTO departments VALUES
(1, 'IT'),
(2, 'HR'),
(3, 'Finance');

INSERT INTO employees VALUES
(1, 'Rahul', 60000, 1),
(2, 'Priya', 75000, 1),
(3, 'Arjun', 45000, 2),
(4, 'Sneha', 80000, 3);

INSERT INTO customers VALUES
(1, 'Customer A', 'Hyderabad'),
(2, 'Customer B', 'Bangalore'),
(3, 'Customer C', 'Chennai');

INSERT INTO orders VALUES
(101, 1, 5000, '2026-01-10'),
(102, 2, 8000, '2026-01-12'),
(103, 1, 3000, '2026-01-15'),
(104, 3, 6000, '2026-01-20');
""")

conn.commit()

print("Database created successfully!")

print("\nEmployees:")
for row in conn.execute("SELECT * FROM employees"):
    print(row)

Database created successfully!

Employees:
(1, 'Rahul', 60000.0, 1)
(2, 'Priya', 75000.0, 1)
(3, 'Arjun', 45000.0, 2)
(4, 'Sneha', 80000.0, 3)


In [26]:
def get_database_schema(connection):

    schema = ""

    tables = connection.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        AND name NOT LIKE 'sqlite_%'
        ORDER BY name
    """).fetchall()

    for (table_name,) in tables:

        columns = connection.execute(
            f'PRAGMA table_info("{table_name}")'
        ).fetchall()

        schema += f"\nTable: {table_name}\n"

        for column in columns:
            schema += (
                f"Column: {column[1]}, "
                f"Type: {column[2]}\n"
            )

    return schema


schema = get_database_schema(conn)

print("Database Schema:")
print(schema)

Database Schema:

Table: customers
Column: customer_id, Type: INTEGER
Column: customer_name, Type: TEXT
Column: city, Type: TEXT

Table: departments
Column: department_id, Type: INTEGER
Column: department_name, Type: TEXT

Table: employees
Column: employee_id, Type: INTEGER
Column: employee_name, Type: TEXT
Column: salary, Type: REAL
Column: department_id, Type: INTEGER

Table: orders
Column: order_id, Type: INTEGER
Column: customer_id, Type: INTEGER
Column: amount, Type: REAL
Column: order_date, Type: TEXT



In [27]:
import re

def generate_sql(question):

    prompt = f"""
You are an expert SQLite query generator.

DATABASE SCHEMA:
{schema}

USER QUESTION:
{question}

RULES:
1. Generate valid SQLite SQL.
2. Use only tables and columns in the schema.
3. Generate only SELECT or WITH queries.
4. Never generate INSERT, UPDATE, DELETE or DROP.
5. Do not generate PRAGMA, ATTACH or multiple statements.
6. Return only the SQL query.
7. Do not include explanations or Markdown.

SQL:
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "Generate accurate SQLite read-only queries."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_completion_tokens=1024
    )

    sql_query = response.choices[0].message.content.strip()

    sql_query = re.sub(
        r"^```(?:sql)?\s*|\s*```$",
        "",
        sql_query,
        flags=re.IGNORECASE
    ).strip()

    return sql_query


question = "Find the employee with the highest salary"

generated_sql = generate_sql(question)

print("Generated SQL:")
print(generated_sql)

Generated SQL:
SELECT employee_id, employee_name, salary
FROM employees
WHERE salary = (SELECT MAX(salary) FROM employees);


In [28]:
import time

def execute_safe_sql(sql_query):

    sql = sql_query.strip()

    if not re.match(r"^(SELECT|WITH)\b", sql, re.IGNORECASE):
        raise ValueError("Only SELECT or WITH queries are allowed.")

    if ";" in sql.rstrip().rstrip(";"):
        raise ValueError("Multiple SQL statements are not allowed.")

    allowed_actions = {
        sqlite3.SQLITE_SELECT,
        sqlite3.SQLITE_READ,
        sqlite3.SQLITE_FUNCTION,
        sqlite3.SQLITE_RECURSIVE
    }

    def safety_guard(action, arg1, arg2, database, trigger):

        if action in allowed_actions:
            return sqlite3.SQLITE_OK

        return sqlite3.SQLITE_DENY

    conn.set_authorizer(safety_guard)

    try:
        start_time = time.perf_counter()

        cursor = conn.execute(sql)

        rows = cursor.fetchall()

        columns = [
            item[0] for item in cursor.description
        ]

        execution_time = time.perf_counter() - start_time

        query_plan = conn.execute(
            "EXPLAIN QUERY PLAN " + sql
        ).fetchall()

        return {
            "columns": columns,
            "rows": rows,
            "execution_time": execution_time,
            "query_plan": query_plan
        }

    finally:

        conn.set_authorizer(None)


result = execute_safe_sql(generated_sql)

print("Columns:", result["columns"])

print("\nQuery Results:")
for row in result["rows"]:
    print(row)

print(
    "\nExecution Time:",
    round(result["execution_time"], 6),
    "seconds"
)

print("\nQuery Execution Plan:")
for item in result["query_plan"]:
    print(item)

Columns: ['employee_id', 'employee_name', 'salary']

Query Results:
(4, 'Sneha', 80000.0)

Execution Time: 0.000191 seconds

Query Execution Plan:
(2, 0, 0, 'SCAN employees')
(8, 0, 0, 'SCALAR SUBQUERY 1')
(13, 8, 0, 'SEARCH employees')


In [29]:
def text_to_sql_pipeline(question):

    print("\n" + "=" * 50)
    print("USER QUESTION:", question)
    print("=" * 50)

    try:

        sql_query = generate_sql(question)

        print("\nGenerated SQL:")
        print(sql_query)


        result = execute_safe_sql(sql_query)

        print("\nQuery Results:")

        print(" | ".join(result["columns"]))

        for row in result["rows"]:
            print(row)

        print(
            "\nExecution Time:",
            round(result["execution_time"], 6),
            "seconds"
        )

        print("\nOptimization Suggestions:")

        suggestions = []

        if re.search(r"\bSELECT\s+\*", sql_query, re.I):
            suggestions.append(
                "Select only the required columns instead of SELECT *."
            )

        if re.search(r"\bJOIN\b", sql_query, re.I):
            suggestions.append(
                "Check indexes on the columns used for joining."
            )

        if any(
            "SCAN " in str(item).upper()
            for item in result["query_plan"]
        ):
            suggestions.append(
                "Review table scans and consider suitable indexes."
            )

        if not suggestions:
            suggestions.append(
                "No obvious optimization suggestions identified."
            )

        for suggestion in suggestions:
            print("-", suggestion)

    except Exception as error:
        print("\nPipeline Error:", error)


text_to_sql_pipeline(
    "Find the employee with the highest salary"
)

text_to_sql_pipeline(
    "Show the average salary in each department"
)

text_to_sql_pipeline(
    "Find customers from Hyderabad"
)

text_to_sql_pipeline(
    "Show total sales for each customer"
)


USER QUESTION: Find the employee with the highest salary

Generated SQL:
SELECT employee_id, employee_name, salary
FROM employees
WHERE salary = (SELECT MAX(salary) FROM employees);

Query Results:
employee_id | employee_name | salary
(4, 'Sneha', 80000.0)

Execution Time: 0.00022 seconds

Optimization Suggestions:
- Review table scans and consider suitable indexes.

USER QUESTION: Show the average salary in each department

Generated SQL:
SELECT d.department_name, AVG(e.salary) AS average_salary
FROM employees e
JOIN departments d ON e.department_id = d.department_id
GROUP BY d.department_id, d.department_name;

Query Results:
department_name | average_salary
('IT', 67500.0)
('HR', 45000.0)
('Finance', 80000.0)

Execution Time: 0.000208 seconds

Optimization Suggestions:
- Check indexes on the columns used for joining.
- Review table scans and consider suitable indexes.

USER QUESTION: Find customers from Hyderabad

Generated SQL:
SELECT * FROM customers WHERE city = 'Hyderabad';

Qu